<p><font size="6" color="grey"><b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font></p>


<p><font size="5" color="grey"><b>
A00 | Code-Snippets Agenten
</b></font></p>

---


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/hands_on.png" class="logo" width="750"/>
<p><font color='black' size="2">
KI-generiertes Bild
</font></p>

<p><font color='blue' size="5">
🧤 Hands-on statt Theorie
</font></p>

Dieses Notebook ist eine kompakte Copy/Paste-Sammlung für Google Colab. Es enthält lauffähige Grundmuster für LangChain, LangGraph, Checkpointing, Human-in-the-Loop, Multi-Agent-Patterns und LangSmith im Agenten-Kurs. Ausführliche Erklärungen stehen im Skript und im Cheatsheet.


# 0 | Colab-Setup & Importe


# 0.1 | Umgebung einrichten


In [ ]:
#@markdown Umgebung einrichten { display-mode: "form" }

# --- Install Kursbibliothek --------------------------------------
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul


# --- LangSmith Tracing ------------------------------------------
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "A00-Snippets-Agenten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# --- LangSmith-Konfiguration -------------------------------------
run_cfg = {
    "run_name": "A00_Snippets_Agenten",
    "tags": ["a00", "snippets"],
    "metadata": {"notebook": "A00", "kurs": "Agenten"},
}

# --- Import Kursbibliothek --------------------------------------
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid,
    show_trace,
)

# --- Setup API-Keys ----------------------------------------------
setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# --- Import Modell-Konfiguration ---------------------------------
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)


# 0.2 | Standard-Imports


In [ ]:
# Standardimporte für Agenten-Notebooks

# ── 1. Python Standard Library ────────────────────────────────────────────────
import os
import re
import sqlite3
from pathlib import Path
from typing import Annotated, Literal

# ── 2. Typing & Datenmodelle ──────────────────────────────────────────────────
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

# ── 3. IPython / Notebook-Ausgabe ─────────────────────────────────────────────
from IPython.display import Markdown, display

# ── 4. LangChain – Modell & Agenten ───────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# ── 5. LangChain – Messages, Prompts, Parser & Tools ─────────────────────────
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# ── 6. LangGraph – State, Checkpointing & Tool-Routing ────────────────────────
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

llm = init_chat_model(BASELINE)
parser = StrOutputParser()

# 1 | Minimaler LangChain-Baustein


In [ ]:
prompt = ChatPromptTemplate([
    ("system", "Antworte kurz, konkret und auf Deutsch."),
    ("user", "{frage}"),
])
chain = prompt | llm | StrOutputParser()

antwort = chain.invoke({"frage": "Was ist ein KI-Agent?"}, config=run_cfg)
print(antwort)


# 2 | Structured Output


In [ ]:
class FrageTyp(BaseModel):
    kategorie: Literal["definition", "retrieval", "out_of_corpus"] = Field(
        description="Kategorie der Research-Frage"
    )

router_llm = llm.with_structured_output(FrageTyp)
ergebnis = router_llm.invoke("Warum verbessert RAG die Zuverlässigkeit?", config=run_cfg)
print(ergebnis.kategorie)


# 3 | Agent mit Tools


In [ ]:
@tool
def quellen_check(thema: str) -> str:
    """Prüft grob, ob ein Thema zum Meeting-Briefing-Korpus passt."""
    kursnah = ["rag", "retrieval", "evaluation", "agent", "embedding"]
    return "Korpusnah" if any(w in thema.lower() for w in kursnah) else "Korpusabdeckung unklar"

agent = create_agent(
    model=init_chat_model(WORKER),
    tools=[quellen_check],
    system_prompt="Du bist ein Meeting- & Research-Briefing-Agent. Nutze Tools für Projektfragen.",
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Prüfe RAG-Evaluation."}]
}, config=run_cfg)
print(response["messages"][-1].content)


# 4 | StateGraph

# 4.1 | Minimaler StateGraph


In [ ]:
class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    routing: str
    antwort: str


def analyse_node(state: ResearchState) -> dict:
    frage = state["messages"][-1].content
    routing = "retrieval" if "rag" in frage.lower() else "definition"
    return {"routing": routing}


def antwort_node(state: ResearchState) -> dict:
    return {"antwort": f"Gewählter Pfad: {state['routing']}"}

builder = StateGraph(ResearchState)
builder.add_node("analyse", analyse_node)
builder.add_node("antwort", antwort_node)

builder.add_edge(START, "analyse")
builder.add_edge("analyse", "antwort")
builder.add_edge("antwort", END)

graph = builder.compile()
result = graph.invoke({
    "messages": [HumanMessage(content="Warum verbessert RAG Antworten?")],
    "routing": "",
    "antwort": "",
}, config=run_cfg)
print(result["antwort"])


# 4.2 | StateGraph visualisieren

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

# 5 | Conditional Routing


In [ ]:
def definition_node(state: ResearchState) -> dict:
    return {"antwort": "Definitionspfad: kurz erklären."}


def retrieval_node(state: ResearchState) -> dict:
    return {"antwort": "Retrievalpfad: Korpus abrufen und Quellen nennen."}


def route_by_category(state: ResearchState) -> str:
    return state["routing"]

builder = StateGraph(ResearchState)
builder.add_node("analyse", analyse_node)
builder.add_node("definition", definition_node)
builder.add_node("retrieval", retrieval_node)

builder.add_edge(START, "analyse")
builder.add_conditional_edges(
    "analyse",
    route_by_category,
    {"definition": "definition", "retrieval": "retrieval"},
)
builder.add_edge("definition", END)
builder.add_edge("retrieval", END)

routing_graph = builder.compile()
result = routing_graph.invoke({
    "messages": [HumanMessage(content="Welche Quellen braucht RAG?")],
    "routing": "",
    "antwort": "",
}, config=run_cfg)
print(result["antwort"])


# 6 | Qualitätsgate mit Schleife


In [ ]:
class QualityState(TypedDict):
    antwort: str
    score: float
    versuche: int


def schreiben_node(state: QualityState) -> dict:
    return {"antwort": state.get("antwort") or "RAG verbessert Antworten durch Retrieval."}


def check_node(state: QualityState) -> dict:
    score = 0.9 if "Quelle:" in state["antwort"] else 0.4
    return {"score": score, "versuche": state["versuche"] + 1}


def revise_node(state: QualityState) -> dict:
    return {"antwort": state["antwort"] + " Quelle: kurs_korpus.md"}


def quality_router(state: QualityState) -> str:
    if state["score"] < 0.7 and state["versuche"] < 2:
        return "revise"
    return END

builder = StateGraph(QualityState)
builder.add_node("schreiben", schreiben_node)
builder.add_node("check", check_node)
builder.add_node("revise", revise_node)
builder.add_edge(START, "schreiben")
builder.add_edge("schreiben", "check")
builder.add_conditional_edges("check", quality_router)
builder.add_edge("revise", "check")
quality_graph = builder.compile()

result = quality_graph.invoke({"antwort": "", "score": 0.0, "versuche": 0}, config=run_cfg)
print(result)


# 7 | Tool-Loop mit ToolNode


In [ ]:
@tool
def research_signal(text: str) -> str:
    """Extrahiert einfache Research-Signale."""
    begriffe = ["rag", "retrieval", "evaluation", "quelle"]
    treffer = [b for b in begriffe if b in text.lower()]
    return ", ".join(treffer) if treffer else "Keine klaren Signale."

tools = [research_signal]
llm_with_tools = init_chat_model(WORKER).bind_tools(tools)

class ToolState(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: ToolState) -> dict:
    system = SystemMessage(content="Du bist ein Meeting- & Research-Briefing-Agent. Nutze Tools bei Projektfragen.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(ToolState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

tool_graph = builder.compile()
result = tool_graph.invoke({
    "messages": [HumanMessage(content="Welche Research-Signale stecken in RAG-Evaluation?")]
}, config=run_cfg)
print(result["messages"][-1].content)


# 8 | Checkpointing & Sessions


In [ ]:
class SessionState(TypedDict):
    messages: Annotated[list, add_messages]


def session_node(state: SessionState) -> dict:
    history = "\n".join(msg.content for msg in state["messages"])
    if "RAG-Evaluation" in history:
        antwort = "Session-Kontext: Thema ist RAG-Evaluation."
    else:
        antwort = "Kein Research-Thema in dieser Session."
    return {"messages": [AIMessage(content=antwort)]}

builder = StateGraph(SessionState)
builder.add_node("session", session_node)
builder.add_edge(START, "session")
builder.add_edge("session", END)
session_graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "briefing-mara-demo"}, **run_cfg}
session_graph.invoke({"messages": [HumanMessage(content="Merke: Thema ist RAG-Evaluation.")]}, config=config)
result = session_graph.invoke({"messages": [HumanMessage(content="Was war das Thema?")]}, config=config)
print(result["messages"][-1].content)


# 9 | Human-in-the-Loop


In [ ]:
class ReviewState(TypedDict):
    entwurf: str
    genehmigt: bool
    finaler_text: str


def review_node(state: ReviewState) -> dict:
    entscheidung = interrupt({
        "frage": "Research-Antwort freigeben?",
        "entwurf": state["entwurf"],
    })
    return {"genehmigt": str(entscheidung).lower().strip() in {"ja", "j", "true"}}


def publish_node(state: ReviewState) -> dict:
    if not state["genehmigt"]:
        return {"finaler_text": "Nicht veröffentlicht."}
    return {"finaler_text": state["entwurf"]}

builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_node("publish", publish_node)
builder.add_edge(START, "review")
builder.add_edge("review", "publish")
builder.add_edge("publish", END)
review_graph = builder.compile(checkpointer=InMemorySaver())

review_config = {"configurable": {"thread_id": "review-demo"}, **run_cfg}
start_state = {
    "entwurf": "RAG-Evaluation braucht Quellenbindung und Qualitätsmessung.",
    "genehmigt": False,
    "finaler_text": "",
}

first = review_graph.invoke(start_state, config=review_config)
if "__interrupt__" in first:
    print(first["__interrupt__"][0].value)
    result = review_graph.invoke(Command(resume="ja"), config=review_config)
    print(result["finaler_text"])


# 10 | Multi-Agent Supervisor


In [ ]:
class SupervisorState(TypedDict):
    frage: str
    route: str
    tabellenbefund: str
    textbefund: str
    synthese: str
    log: list[str]


def supervisor_node(state: SupervisorState) -> dict:
    if not state["tabellenbefund"]:
        route = "tabellen_worker"
    elif not state["textbefund"]:
        route = "text_worker"
    elif not state["synthese"]:
        route = "synthese_worker"
    else:
        route = "FINISH"
    return {"route": route, "log": state["log"] + [f"Supervisor -> {route}"]}


def supervisor_router(state: SupervisorState) -> str:
    return END if state["route"] == "FINISH" else state["route"]


def tabellen_worker(state: SupervisorState) -> dict:
    return {"tabellenbefund": "Precision@3 = 0.77", "log": state["log"] + ["Tabellen-Worker"]}


def text_worker(state: SupervisorState) -> dict:
    return {"textbefund": "Quellenpflicht senkt Halluzinationsrisiken.", "log": state["log"] + ["Text-Worker"]}


def synthese_worker(state: SupervisorState) -> dict:
    return {"synthese": "Antwort mit Kennzahl und Quellenpflicht.", "log": state["log"] + ["Synthese-Worker"]}

builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("tabellen_worker", tabellen_worker)
builder.add_node("text_worker", text_worker)
builder.add_node("synthese_worker", synthese_worker)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "tabellen_worker": "tabellen_worker",
        "text_worker": "text_worker",
        "synthese_worker": "synthese_worker",
        END: END,
    },
)
for node_name in ["tabellen_worker", "text_worker", "synthese_worker"]:
    builder.add_edge(node_name, "supervisor")

supervisor_graph = builder.compile()
result = supervisor_graph.invoke({
    "frage": "Wie bewerte ich RAG?",
    "route": "",
    "tabellenbefund": "",
    "textbefund": "",
    "synthese": "",
    "log": [],
}, config=run_cfg)
print(result["synthese"])
print(result["log"])


# 11 | LangSmith


# 11.1 | Run-Konfiguration - Einzelaufruf Chain


In [ ]:
# LangSmith-Konvention für einzelne Snippets und Graph-Runs
langsmith_run_cfg = {
    "run_name": "A00_LangSmith_Kurzantwort",
    "tags": ["a00", "langsmith", "snippet"],
    "metadata": {
        "kurs": "Agenten",
        "notebook": "A00_snippets_agenten",
        "baustein": "langsmith-run-config",
    },
}

antwort = chain.invoke(
    {"frage": "Warum ist LangSmith für Agenten hilfreich?"},
    config=langsmith_run_cfg,
)
print(antwort)

# 11.2 | with_config() - in Chain konfiguriert


In [ ]:
traced_chain = (prompt | llm | StrOutputParser()).with_config({
    "run_name": "A00_Traced_Chain",
    "tags": ["a00", "with_config"],
    "metadata": {"komponente": "prompt-llm-parser"},
})

antwort = traced_chain.invoke({"frage": "Was zeigt ein LangSmith-Trace?"})
print(antwort)

# 11.3 | Trace mit thread_id für Sessions


In [ ]:
trace_config = {
    "configurable": {"thread_id": "a00-langsmith-session"},
    "run_name": "A00_Session_Trace",
    "tags": ["a00", "checkpointing", "langsmith"],
    "metadata": {"workflow": "session-demo"},
}

session_graph.invoke(
    {"messages": [HumanMessage(content="Merke: LangSmith zeigt Node-Schritte.")]},
    config=trace_config,
)
result = session_graph.invoke(
    {"messages": [HumanMessage(content="Was zeigt LangSmith?")]},
    config=trace_config,
)
print(result["messages"][-1].content)


# 11.4 | Mini-Evaluation vorbereiten


In [ ]:
# Optional: Dataset/Evaluation in LangSmith vorbereiten.
# Diese Zelle erzeugt echte LangSmith-Objekte und braucht LANGSMITH_API_KEY.

try:
    from langsmith import Client
    from langsmith.evaluation import evaluate

    client = Client(api_url=os.environ["LANGSMITH_ENDPOINT"])
    dataset_name = "A00 Meeting-Briefing-Agent Smoke Test"

    try:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="Kleine Regressionstests für Agenten-Snippets.",
        )
        client.create_example(
            inputs={"frage": "Wann ist LangGraph sinnvoll?"},
            outputs={"must_contain": "Routing"},
            dataset_id=dataset.id,
        )
    except Exception as exc:
        print(f"Dataset existiert vermutlich bereits oder konnte nicht angelegt werden: {exc}")

    def target(inputs: dict) -> dict:
        antwort = traced_chain.invoke({"frage": inputs["frage"]})
        return {"antwort": antwort}

    def contains_expected(outputs: dict, reference_outputs: dict) -> bool:
        return reference_outputs["must_contain"].lower() in outputs["antwort"].lower()

    print("Evaluation vorbereitet. Ausführen bei Bedarf:")
    print("evaluate(target, data=dataset_name, evaluators=[contains_expected], experiment_prefix='a00-smoke')")
except Exception as exc:
    print(f"LangSmith-Evaluation übersprungen: {exc}")


# 11.5 | Trace anzeigen


In [ ]:
# Optional: aktuelle Runs im LangSmith-Projekt anzeigen
# show_trace("A00-Snippets-Agenten", limit=3, show_steps=True)

print("LangSmith ist über run_cfg, langsmith_run_cfg und trace_config vorbereitet.")


# 12 | LLM-Caching


In [ ]:
# InMemoryCache speichert Antworten identischer Modellaufrufe im aktuellen Prozess.
set_llm_cache(InMemoryCache())

cached_response = llm.invoke("Erkläre Tool-Use bei Agenten in einem Satz.")
print(cached_response.content)

# Caching ist kein Memory: Es speichert Antworten zu gleichen Requests,
# aber keinen Gesprächs- oder Graph-Zustand.
# Agenten- und Tool-Läufe mit Seiteneffekten nicht blind cachen.
